# Experiment: Prediction-Only Expert Identification

This notebook runs the full method and baselines over a set of prediction JSONs
and writes one results record per (dataset, size, trial) to a JSONL file. It is
intentionally thin: all logic lives in the importable modules under `src/`.

**Inputs:** a directory of prediction JSONs, one per dataset, each named
`<dataset>_test_predictions_logits.json` and following the schema documented in
`src/io_utils.py`.

**Output:** a JSONL file of run records (the input to the analysis notebooks).
Each record contains, per run, the macro-F1 of every method, the expert
partition, per-class prediction-only signals, per-model class-conditional F1
(used by the plots), and timing. No raw datasets are needed to run this; only
the prediction JSONs.


In [ ]:
# If running from the repo root, make src/ importable.
import sys, os
sys.path.append(os.path.join(os.getcwd(), "src"))

import json
import glob

from io_utils import load_dataset_predictions, make_test_sizes
from pipeline import run_dataset

## Configuration

Point `PREDICTIONS_DIR` at the folder containing the prediction JSONs, and
`OUTPUT_JSONL` at where the run records should be written. The protocol
constants below match the paper (subset-size range [3, 10], 20 evaluation
sizes, 3 trials per size). The model pool size is read from each JSON.

In [ ]:
# --- paths (edit these) ---
PREDICTIONS_DIR = "data/predictions"          # folder of <dataset>_test_predictions_logits.json
OUTPUT_JSONL    = "data/results/experiment_records.jsonl"

# --- protocol constants (match the paper) ---
NUM_SIZES        = 20      # evaluation sizes swept per dataset
TRIALS_PER_SIZE  = 3       # independent trials per size
MIN_SUBSET_SIZE  = 3       # Dunn search lower bound (r_min)
MAX_SUBSET_SIZE  = 10      # Dunn search upper bound (r_max)
ENFORCE_NUM_MODELS = 15    # expected pool size; set None to disable the check

os.makedirs(os.path.dirname(OUTPUT_JSONL), exist_ok=True)

## Run

For each prediction JSON, we load it, build the evaluation-size schedule from
the stored `N_test_min` and the available test size, run the full size/trial
sweep, and append the resulting records to the JSONL file. A reviewer can run
this on a handful of representative datasets and compare the per-method macro-F1
against the paper's tables.

In [ ]:
json_paths = sorted(glob.glob(os.path.join(PREDICTIONS_DIR, "*_test_predictions_logits.json")))
print(f"Found {len(json_paths)} prediction file(s) in {PREDICTIONS_DIR}")

# Truncate (overwrite) the output file at the start of a fresh run.
open(OUTPUT_JSONL, "w").close()

summaries = []
for path in json_paths:
    loaded = load_dataset_predictions(path, enforce_num_models=ENFORCE_NUM_MODELS)

    test_sizes = make_test_sizes(
        min_size=loaded["n_test_min"],
        max_size=len(loaded["y_true"]),
        num_sizes=NUM_SIZES,
    )

    records = run_dataset(
        loaded,
        test_sizes=test_sizes,
        trials_per_size=TRIALS_PER_SIZE,
        min_subset_size=MIN_SUBSET_SIZE,
        max_subset_size=MAX_SUBSET_SIZE,
    )

    with open(OUTPUT_JSONL, "a") as f:
        for rec in records:
            f.write(json.dumps(rec) + "\n")

    summaries.append((loaded["dataset"], len(records)))
    print(f"  {loaded['dataset']}: {len(records)} records "
          f"({len(test_sizes)} sizes x {TRIALS_PER_SIZE} trials)")

print("\nDone.")
for name, n in summaries:
    print(f"  {name}: {n} records")

## Quick check

Print the per-method mean macro-F1 across all runs of each dataset, as a sanity
check against the paper's results tables.

In [ ]:
import collections

by_dataset = collections.defaultdict(lambda: collections.defaultdict(list))
with open(OUTPUT_JSONL) as f:
    for line in f:
        rec = json.loads(line)
        for method, f1 in rec["macro_f1"].items():
            by_dataset[rec["dataset"]][method].append(f1)

for dataset, methods in by_dataset.items():
    print(f"\n{dataset}")
    for method, vals in methods.items():
        print(f"  {method:18s} mean macro-F1 = {sum(vals)/len(vals):.4f}  (n={len(vals)})")